<a href="https://colab.research.google.com/github/nattagon256312-afk/RL-watering/blob/main/RL_Train_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install stable-baselines3[extra]
!pip install stable-baselines3[extra] tf2onnx tensorflow
!pip install onnxscript onnx

In [ ]:
import numpy as np
import random
import os
import torch
import tensorflow as tf

from gymnasium import Env
from gymnasium.spaces import Discrete, Box
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy

In [ ]:
class wateringEnv(Env):
  def __init__(self):
    self.action_space = Discrete(6)
    self.water_action = [0,20,40,60,80,100]
    self.observation_space = Box(low=-1.0, high=1.0, shape=(7,), dtype=np.float32)
    self.alpha = 0.5
    self.max_water = 800
    self.max_cap = 1000
    self.episode_days = 7
    self.max_steps = self.episode_days * 24

  def _get_obs(self):
    hour = self.time % 24
    time_remaining = self.time_length / self.max_steps

    return np.array([self.humidity / 1024,
                           self.water_total / self.max_cap,
                           self.max_water / self.max_cap,
                           time_remaining,
                           np.sin(2*np.pi*hour/24),
                           np.cos(2*np.pi*hour/24),
                           min(self.last_e/20, 1.0)], dtype=np.float32)


  def step(self, action):
    water = self.water_action[action]
    self.evap_state = 0.8*self.evap_state + random.gauss(0, 1.0)
    hour = self.time % 24
    daily = 1.0 + 0.5*np.cos(2*np.pi*(hour-12)/24)
    e_t = max(0.0, self.evap_scale * daily * (6.0 + self.evap_state))
    self.last_e = e_t

    water_loss= e_t*1
    actual_water = min(water, self.water_total)

    self.humidity = self.humidity + self.alpha*actual_water - water_loss
    self.humidity = np.clip(self.humidity, 0, 1024)
    self.water_total -= actual_water



    self.time_length -= 1
    self.time += 1

    low, high = 500, 750
    if self.humidity < low:
      reward = -(low - self.humidity) / 1024
    elif self.humidity > high:
      reward = -0.5 * (self.humidity - high) / 1024
    else:
      reward = 0.1

    reward -= 0.01 * (actual_water / 100)



    terminated = False
    truncated = self.time_length <= 0

    if self.water_total <= 0 and self.time_length > 0:
      reward -= 0.2

    return self._get_obs(), reward, terminated, truncated, {}

  def reset(self, seed=None, options=None):
    super().reset(seed=seed)
    self.episode_days = random.choice([3, 5, 7, 10, 14])
    self.max_steps = self.episode_days * 24
    self.time_length = self.max_steps
    self.max_water = random.randint(300, 1000)
    self.water_total = self.max_water
    self.humidity = float(random.randint(0,1024))
    self.time = 0
    self.evap_scale = random.uniform(0.5, 2.0)
    self.evap_state = 0.0
    self.last_e = 0.0

    return self._get_obs(), {}
















In [ ]:
episodes = 5
env = wateringEnv()
for episode in range(1, episodes+1):
  state, info = env.reset()
  done = False
  score = 0

  while not done:
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    score += reward
  print (f'Episode:{episode} Score:{score}')

In [ ]:
log_path = "./ppo_watering_tensorboard/"
model = PPO('MlpPolicy', env, verbose=1, tensorboard_log=log_path)
model.learn(total_timesteps=1000000)



In [ ]:

import torch.nn as nn

class ExtractPPOActor(nn.Module):
    def __init__(self, policy):
        super().__init__()
        self.policy = policy

    def forward(self, obs):
        features = self.policy.features_extractor(obs)
        latent_pi, _ = self.policy.mlp_extractor(features)
        action_logits = self.policy.action_net(latent_pi)

        return action_logits


pure_actor = ExtractPPOActor(model.policy)
pure_actor.eval()


dummy_input = torch.randn(1, 7, dtype=torch.float32)

torch.onnx.export(
    pure_actor,
    dummy_input,
    "ppo_actor.onnx",
    input_names=["input"],
    output_names=["output"],
    opset_version=11,
    dynamo=False
)



In [ ]:

!pip install onnx2tf


!onnx2tf -i ppo_actor.onnx -o ppo_tf_output


!cp ppo_tf_output/ppo_actor_float32.tflite rl_model.tflite
!xxd -i rl_model.tflite > rl_model.h

